In [ ]:
# ==============================================================================
# File: 01_Prepare_Data.ipynb (Cell 1 of 5)
# What this notebook does: Automates end-to-end cloud dataset acquisition, zero-leakage
#   Leave-One-Generator-Out (LOGO) splitting, and GPU-accelerated MTCNN face cropping
#   on Google Colab's fast local SSD, then backs up the artifacts to Google Drive.
# Inputs: User's Google Drive account, kaggle.json API token, and the ZeroFake repository.
# Outputs: Saves cropped_faces.zip, splits.json, frames_manifest.csv, and
#   failed_detections.csv permanently into /content/drive/MyDrive/ZeroFake_Storage/.
# ==============================================================================

# --- CELL 1: Mount Google Drive & Create Persistent Storage Folder ---
import os
from google.colab import drive

# 1. Mount your Google Drive at /content/drive
drive.mount('/content/drive')

# 2. Create the permanent storage folder on Google Drive if it does not exist yet
STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
os.makedirs(STORAGE_DIR, exist_ok=True)
print(f'[SUCCESS] Google Drive mounted. Persistent storage ready at: {STORAGE_DIR}')


In [ ]:
# --- CELL 2: Upload kaggle.json Credential & Install Dependencies ---
# Plain-English explanation:
#   Prompts you to select your downloaded kaggle.json API token from your computer,
#   places it in ~/.kaggle/kaggle.json with strict owner-only read permissions (chmod 600),
#   and installs all required Python libraries from requirements.txt.

import os
import shutil
import subprocess
from google.colab import files

KAGGLE_DIR = os.path.expanduser('~/.kaggle')
KAGGLE_JSON_PATH = os.path.join(KAGGLE_DIR, 'kaggle.json')
os.makedirs(KAGGLE_DIR, exist_ok=True)

if not os.path.exists(KAGGLE_JSON_PATH):
    print('[ACTION REQUIRED] Please select and upload your kaggle.json file below:')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('kaggle.json was not uploaded. Please re-run this cell and select kaggle.json.')
    shutil.move('kaggle.json', KAGGLE_JSON_PATH)

# Lock file permissions to 600 (required by the Kaggle CLI so your token stays private)
os.chmod(KAGGLE_JSON_PATH, 0o600)
print(f'[SUCCESS] Configured Kaggle token at {KAGGLE_JSON_PATH} (chmod 600).')

# Install required project packages from requirements.txt (plus kaggle CLI)
if os.path.exists('requirements.txt'):
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt', 'kaggle'], check=True)
else:
    subprocess.run([
        'pip', 'install', '-q', 'kaggle', 'torch', 'torchvision', 'timm',
        'facenet-pytorch', 'opencv-python-headless', 'scikit-learn',
        'matplotlib', 'seaborn', 'grad-cam', 'gradio', 'pyyaml', 'tqdm'
    ], check=True)
print('[SUCCESS] Installed all ZeroFake dependencies from requirements.txt.')


In [ ]:
# --- CELL 3: Download FaceForensics++ (c23) to Colab's Fast Local SSD ---
# Plain-English explanation:
#   Downloads the pre-packaged FaceForensics++ c23 dataset ('xdxd003/ff-c23') directly
#   from Kaggle onto Colab's high-speed local SSD (/content/raw_data/ffpp) and unzips it.
#   Nothing is downloaded to your laptop, and local SSD avoids Google Drive network latency.

import os
import subprocess
import zipfile

RAW_FFPP_DIR = '/content/raw_data/ffpp'
os.makedirs(RAW_FFPP_DIR, exist_ok=True)

print('[INFO] Downloading FaceForensics++ (c23) dataset (xdxd003/ff-c23) via Kaggle API...')
subprocess.run([
    'kaggle', 'datasets', 'download', '-d', 'xdxd003/ff-c23', '-p', RAW_FFPP_DIR
], check=True)

zip_archive = os.path.join(RAW_FFPP_DIR, 'ff-c23.zip')
if os.path.exists(zip_archive):
    print(f'[INFO] Unzipping {zip_archive} onto local Colab SSD ({RAW_FFPP_DIR})...')
    with zipfile.ZipFile(zip_archive, 'r') as zf:
        zf.extractall(RAW_FFPP_DIR)
    os.remove(zip_archive)
    print('[SUCCESS] Unzipped FaceForensics++ (c23) and removed temporary raw zip archive.')
else:
    print('[INFO] Dataset already extracted inside /content/raw_data/ffpp.')


In [ ]:
# --- CELL 4: Run Zero-Leakage LOGO Splitter & MTCNN Face Cropper ---
# Plain-English explanation:
#   1. Runs preprocessing/create_splits.py holding out 'NeuralTextures' 100% for test_unseen,
#      while grouping swapped actor identity IDs (like 000_003.mp4) so no person's face ever
#      leaks across train (80%), val (10%), or test_closed (10%).
#   2. Runs preprocessing/extract_and_crop.py on GPU to sample 16 evenly spaced frames per video,
#      detect and align faces with MTCNN (1.3x margin), and save 224x224 RGB crops into /content/processed_faces.

import os
import subprocess

RAW_FFPP_DIR = '/content/raw_data/ffpp'
PROCESSED_DIR = '/content/processed_faces'
SPLITS_PATH = os.path.join(PROCESSED_DIR, 'splits.json')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Step 4.1: Generate and verify zero-leakage Leave-One-Generator-Out (LOGO) splits
print('[INFO] Running preprocessing/create_splits.py (holding out NeuralTextures)...')
subprocess.run([
    'python', 'preprocessing/create_splits.py',
    '--data_dir', RAW_FFPP_DIR,
    '--output_path', SPLITS_PATH,
    '--holdout_generator', 'NeuralTextures'
], check=True)

# Step 4.2: Extract 16 frames/video, detect & align with MTCNN (1.3x margin), and save 224x224 crops
# Note: We pass --zip_path '' here because Cell 5 explicitly zips /content/processed_faces into /content/cropped_faces.zip
print('[INFO] Running preprocessing/extract_and_crop.py on GPU...')
subprocess.run([
    'python', 'preprocessing/extract_and_crop.py',
    '--data_dir', RAW_FFPP_DIR,
    '--splits_path', SPLITS_PATH,
    '--output_dir', PROCESSED_DIR,
    '--num_frames', '16',
    '--margin_scale', '1.3',
    '--image_size', '224',
    '--zip_path', ''
], check=True)
print(f'[SUCCESS] Saved cropped 224x224 faces and manifests into {PROCESSED_DIR}.')


In [ ]:
# --- CELL 5: Zip Processed Faces & Copy All Artifacts to Google Drive ---
# Plain-English explanation:
#   Packages /content/processed_faces into a single /content/cropped_faces.zip archive
#   (so future training runs can unzip 60,000+ crops to local SSD in seconds without Google Drive
#   file-by-file latency) and copies cropped_faces.zip, splits.json, frames_manifest.csv,
#   and failed_detections.csv permanently into /content/drive/MyDrive/ZeroFake_Storage/.

import os
import shutil
import zipfile
from pathlib import Path

PROCESSED_DIR = '/content/processed_faces'
LOCAL_ZIP_PATH = '/content/cropped_faces.zip'
STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
os.makedirs(STORAGE_DIR, exist_ok=True)

# 1. Zip /content/processed_faces into /content/cropped_faces.zip
print(f'[INFO] Packaging {PROCESSED_DIR} into {LOCAL_ZIP_PATH}...')
processed_root = Path(PROCESSED_DIR)
with zipfile.ZipFile(LOCAL_ZIP_PATH, mode='w', compression=zipfile.ZIP_STORED) as zf:
    for file_path in sorted(processed_root.rglob('*')):
        if file_path.is_file():
            arcname = file_path.relative_to(processed_root).as_posix()
            zf.write(str(file_path), arcname=arcname)

# 2. Copy cropped_faces.zip, splits.json, frames_manifest.csv, and failed_detections.csv to Google Drive
artifacts_to_copy = [
    (LOCAL_ZIP_PATH, os.path.join(STORAGE_DIR, 'cropped_faces.zip')),
    (os.path.join(PROCESSED_DIR, 'splits.json'), os.path.join(STORAGE_DIR, 'splits.json')),
    (os.path.join(PROCESSED_DIR, 'frames_manifest.csv'), os.path.join(STORAGE_DIR, 'frames_manifest.csv')),
    (os.path.join(PROCESSED_DIR, 'failed_detections.csv'), os.path.join(STORAGE_DIR, 'failed_detections.csv')),
]

for src_path, dst_path in artifacts_to_copy:
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        size_mb = os.path.getsize(dst_path) / (1024 * 1024)
        print(f'[COPIED] {os.path.basename(dst_path)} -> {dst_path} ({size_mb:.2f} MB)')
    else:
        raise FileNotFoundError(f'Expected artifact missing before backup: {src_path}')

print('[SUCCESS] All ZeroFake data preparation artifacts are safely stored in Google Drive!')
